# Notebook 06: Panoptic FPN

**Module:** 09 Instance Segmentation  
**Duration:** ~2.5 hrs

---

## Learning Objectives

1. Understand Panoptic FPN multi-task design
2. Know semantic + instance branch fusion
3. Handle overlapping predictions at merge step

## Panoptic FPN (Kirillov et al., 2019)

**Architecture:** Shared FPN backbone with two heads:

1. **Semantic head:** Per-pixel class prediction (stuff + things classes)
2. **Instance head:** Mask R-CNN style for things only

**Fusion (panoptic merge):**
1. Take instance masks (things) sorted by confidence
2. Paste onto canvas (higher confidence wins overlaps)
3. Fill remaining pixels with semantic stuff predictions

**Heuristic:** Instance masks override semantic for thing classes.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import torch
import torch.nn as nn
import torch.nn.functional as F

plt.rcParams['figure.figsize'] = (8, 5)
torch.manual_seed(42)
rng = np.random.default_rng(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
# Panoptic merge heuristic (concept)
H, W = 64, 64
semantic = torch.randint(0, 5, (H, W))  # includes thing classes
inst_masks = [torch.zeros(H, W) for _ in range(3)]
inst_masks[0][10:25, 10:25] = 1
inst_masks[1][30:45, 30:45] = 1
inst_masks[2][15:35, 40:55] = 1
scores = [0.9, 0.85, 0.7]

panoptic_out = semantic.clone()
for mask, score in sorted(zip(inst_masks, scores), key=lambda x: -x[1]):
    panoptic_out[mask.bool()] = 10 + scores.index(score)  # thing instance IDs
print('Unique panoptic labels:', panoptic_out.unique().tolist())

## Limitations

- Merge heuristics can fail on crowded scenes
- Two separate training objectives
- Mask2Former (Notebook 07) unifies with query-based approach

---

## Summary

Panoptic FPN fuses Mask R-CNN instances with semantic FPN predictions.

**Next:** [07_Mask2Former.ipynb](07_Mask2Former.ipynb)